# R2E Quickstart Guide

Welcome to the R2E (Repository to Environment) quickstart guide! This notebook will walk you through the basic steps of using R2E to convert a GitHub repository into an executable environment, generate equivalence tests, and evaluate code modifications.

## Prerequisites

1. Installed R2E and its dependencies.
2. You are in the environment where you installed R2E.
3. This guide uses `python-graphs` as an example repository. Please clone and install it with the following commands.

    1. Install graphviz
        ```bash
        // linux:
        sudo apt-get install python-dev graphviz libgraphviz-dev pkg-config

        // macos:
        brew install graphviz
        ```
    
    2. Install the repo
        ```bash
        mkdir ~/buckets/myrepos/
        cd ~/buckets/myrepos/
        git clone https://github.com/google-research/python-graphs
        cd python-graphs
        pip install -e .
        ```

3. Start `r2e-test-server` in a separate terminal with the command: `r2e-test-server start`

In [57]:
import os
import json
import rpyc
from rich.panel import Panel
from rich.text import Text
from rich.text import Text
from rich.align import Align
from rich.markdown import Markdown
from rich.progress import Progress
from rich.table import Table
from rich.layout import Layout

from r2e.utils.data import load_functions_under_test
from r2e.paths import EXTRACTED_DATA_DIR, TESTGEN_DIR

from helpers import run_command, summarize, console

## Step 1: Setup and Configuration

First, let's find a target repository and set our configurations.

Note: You are free to do this outside of the notebook. All you need to provide is a repository path where your repo is cloned and installed. For now we will use a real-world repository for this guide.

In [ ]:
# Configuration
LOCAL_REPO_PATH = os.path.expanduser("~/buckets/my_repos/python-graphs") # <<< where you cloned and installed your repo!
EXP_ID = "quickstart"
MULTIPROCESS = 8
CACHE_BATCH_SIZE = 100

console.print(Panel(Text("Configuration set!", style="bold green")))

## Step 2: Clone and Setup Repository

Now, let's clone the repository and setup R2E's workspace -- `~/buckets/local_repoeval_bucket/`.

In [ ]:
# clear the workspace
run_command("rm -rf ~/buckets/local_repoeval_bucket/")

# check if repo exists
if not os.path.exists(LOCAL_REPO_PATH):
    raise Exception(f"Repository does not exist at {LOCAL_REPO_PATH}")
else:
    console.print(Panel(Text("Repository cloned!", style="bold green")))

# R2E set's up its workspace to mangle with code
setup_command = f"python -m r2e.repo_builder.setup_repos --local_repo_path {LOCAL_REPO_PATH}"
output = run_command(setup_command)
console.print(Panel(Text("Repository set up in R2E workspace!", style="bold green")))

## Step 3: Extract Functions and Methods

Next, we'll extract functions and methods from the repository.

In [ ]:
extract_command = f"python -m r2e.repo_builder.extract_func_methods --exp_id {EXP_ID} --overwrite_extracted"
output = run_command(extract_command)
summarize(3, "Extract functions and methods", EXP_ID)

## Step 4: Generate Equivalence Tests

Now, let's generate equivalence tests for the extracted functions and methods.

In [ ]:
generate_command = f"python -m r2e.generators.testgen.generate -i {EXP_ID}_extracted.json --exp_id {EXP_ID} --multiprocess {MULTIPROCESS} --use_cache --cache_batch_size {CACHE_BATCH_SIZE}"
output = run_command(generate_command)
console.print(Panel(Text("Equivalence tests generated!", style="bold green")))

##### Let's look at an example function in the repo: `get_control_flow_graph`

In [ ]:
futs = load_functions_under_test(TESTGEN_DIR / f"{EXP_ID}_generate.json")
FUT = [f for f in futs if f.name == "add_node"][0]

# show the function under test in markdown python
code = f"""
```python
{FUT.code}
```
"""
console.print(Markdown(code))

##### R2E automatically extracts the *Dependency Slice* for the function under test.

This serves as context for the LLM-based equivalence test generation process. This is a program slice that contains a topologically sorted
list of code elements that the function under test depends on. 

Note the slice can be quite large, so set `VIEW` to `True` to view the slice.

In [89]:
VIEW = False
if VIEW:
    context = f"""{FUT.context.context}"""
    console.print(Markdown(context))

##### Now, let's look at the generated equivalence tests for the function `get_control_flow_graph`

With these function-test pairs, one can create benchmarks to evaluate the correctness of various LLM4Code tasks such as code generation and code modification.

In [ ]:
code = f"""
```python
{FUT.test_history.latest_tests["test_0"]}
```
"""
console.print(Markdown(code))

## Step 5: Execute Equivalence Tests and Improve Iteratively

Finally, let's execute the equivalence tests and prepare a benchmark.

In [ ]:
def get_service():
    conn = rpyc.connect("localhost", 3006)
    return conn.root

def create_exec_query(fut):
    name, file_path = fut.execution_fut_data
    repo_data = json.dumps({"repo_id": None, "repo_path": fut.repo.repo_path})
    function_data = json.dumps({"funclass_names": [name], "file_path": file_path})
    tests = json.dumps({"generated_tests": fut.tests})
    return repo_data, function_data, tests

def init_service(repo_data, function_data, tests):
    service.setup_repo(repo_data)
    service.setup_function(function_data)
    service.setup_test(tests)
    init_response = service.init()
    init_output = str(init_response["output"])
    init_error = str(init_response["error"])
    
    print(init_output)
    print(init_error)
    

# Step 1: Initialize the service
service = get_service()

# Step 2: Setup the repository, function and tests
futs = load_functions_under_test(TESTGEN_DIR / f"{EXP_ID}_generate.json")
FUT = [f for f in futs if f.name == "add_node"][0]
repo_data, function_data, tests = create_exec_query(FUT)
init_service(repo_data, function_data, tests)

# Step 3: Execute the equivalence tests
out = service.submit()
summarize(6, "Summarize Execution Results", EXP_ID, execution_output=out)

##### R2E also allows you to improve the tests generated by the LLM iteratively w/ the collected execution feedback.

In [206]:
# TODO: add the code to run test repair here